SECTION 1: SETUP AND LOAD PROCESSED DATA

In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import KNNImputer

import matplotlib.pyplot as plt
import seaborn as sns


SECTION 2: CONFIGURATION

In [2]:
DATA_DIR = r"C:\Users\snehi\OneDrive - California State University Chico\MATH699P\Data"
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed_data')

print(f"Input Directory: {PROCESSED_DIR}")
print("\nLoading processed data...")
print("-" * 80)

try:
    df_ozone = pd.read_csv(
        os.path.join(PROCESSED_DIR, 'ozone_processed.csv'),
        parse_dates=['DATE_TIME']
    )
    print(f"Loaded ozone data: {df_ozone.shape}")
except Exception as e:
    print(f"Error loading ozone data: {str(e)}")
    print("Make sure you ran the data extraction notebook first!")
    raise



Input Directory: C:\Users\snehi\OneDrive - California State University Chico\MATH699P\Data\processed_data

Loading processed data...
--------------------------------------------------------------------------------
Loaded ozone data: (21667030, 10)


In [3]:

# Load other datasets
try:
    df_gas = pd.read_csv(
        os.path.join(PROCESSED_DIR, 'hourly_gas_processed.csv'),
        parse_dates=['DATE_TIME']
    )
    print(f"Loaded gas data: {df_gas.shape}")
except:
    df_gas = pd.DataFrame()
    print("No gas data available")

try:
    df_site = pd.read_csv(os.path.join(PROCESSED_DIR, 'site_metadata.csv'))
    print(f"Loaded site metadata: {df_site.shape}")
except:
    df_site = pd.DataFrame()
    print("No site metadata available")



Loaded gas data: (922722, 27)
Loaded site metadata: (159, 19)


SECTION 3: TEMPORAL FEATURE ENGINEERING

In [4]:
class TemporalFeatureEngineer:
    """Create time-based features"""
    
    def __init__(self):
        pass
    
    def add_temporal_features(self, df, datetime_col='DATE_TIME'):
        """Add comprehensive temporal features"""
        
        print("\nAdding temporal features...")
        print("-" * 80)
        
        df = df.copy()
        
        # Ensure datetime
        if datetime_col not in df.columns:
            print(f"Column {datetime_col} not found")
            return df
            
        df[datetime_col] = pd.to_datetime(df[datetime_col])
        
        # Basic temporal features
        df['year'] = df[datetime_col].dt.year
        df['month'] = df[datetime_col].dt.month
        df['day'] = df[datetime_col].dt.day
        df['hour'] = df[datetime_col].dt.hour
        df['dayofweek'] = df[datetime_col].dt.dayofweek  # Monday=0, Sunday=6
        df['dayofyear'] = df[datetime_col].dt.dayofyear
        df['week'] = df[datetime_col].dt.isocalendar().week.astype(int)
        df['quarter'] = df[datetime_col].dt.quarter
        
        # Weekend indicator
        df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
        
        # Time of day categories
        df['hour_category'] = pd.cut(
            df['hour'],
            bins=[-1, 6, 12, 18, 24],
            labels=['night', 'morning', 'afternoon', 'evening']
        )
        
        # Rush hour indicator
        df['is_rush_hour'] = df['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
        
        # Season
        df['season'] = df['month'].map({
            12: 'winter', 1: 'winter', 2: 'winter',
            3: 'spring', 4: 'spring', 5: 'spring',
            6: 'summer', 7: 'summer', 8: 'summer',
            9: 'fall', 10: 'fall', 11: 'fall'
        })
        
        # Cyclical encoding for periodic features
        # Hour (24-hour cycle)
        df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
        
        # Month (12-month cycle)
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
        
        # Day of week (7-day cycle)
        df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
        df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
        
        # Day of year (365-day cycle)
        df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365.25)
        df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365.25)
        
        new_features = [c for c in df.columns if c not in [datetime_col, 'SITE_ID', 'OZONE']]
        print(f"Added {len(new_features)} temporal features")
        
        return df


SECTION 4: LAG AND ROLLING FEATURES

In [5]:
class LagRollingFeatureEngineer:
    """Create lag and rolling window features"""
    
    def __init__(self):
        pass
    
    def create_lag_features(self, df, target_col, lags=[1, 2, 3, 6, 12, 24], 
                          group_col='SITE_ID'):
        """Create lag features"""
        
        if target_col not in df.columns:
            print(f"Target column {target_col} not found")
            return df
        
        print(f"\nCreating lag features for {target_col}...")
        print(f"   Lags: {lags}")
        print("-" * 80)
        
        df = df.copy()
        
        if group_col in df.columns:
            for lag in lags:
                col_name = f'{target_col}_lag_{lag}'
                df[col_name] = df.groupby(group_col)[target_col].shift(lag)
        else:
            for lag in lags:
                col_name = f'{target_col}_lag_{lag}'
                df[col_name] = df[target_col].shift(lag)
            
        print(f"Created {len(lags)} lag features")
        
        return df
    
    def create_diff_features(self, df, target_col, periods, group_col='SITE_ID'):
    # REMOVE THIS LINE: df = df.copy() 
    
        if group_col in df.columns:
            for period in periods:
                # Create the column directly
                col_name = f'{target_col}_diff_{period}'
                # This calculation is much lighter than copying the whole DF
                df[col_name] = df.groupby(group_col)[target_col].diff(periods=period)
                
                # Optimization: Downcast immediately to float32 to save 50% memory
                df[col_name] = df[col_name].astype('float32')
                
        return df


    def create_rolling_features(self, df, target_col, windows=[3, 6, 12, 24, 168],
                               group_col='SITE_ID'):
        """Create rolling window statistics"""
        
        if target_col not in df.columns:
            print(f"Target column {target_col} not found")
            return df
        
        print(f"\nCreating rolling features for {target_col}...")
        print(f"   Windows: {windows}")
        print("-" * 80)
        
                
        if group_col in df.columns:
            for window in windows:
                # Rolling mean
                df[f'{target_col}_rolling_mean_{window}'] = df.groupby(group_col)[target_col].transform(
                    lambda x: x.rolling(window=window, min_periods=1).mean()
                )
                
                # Rolling std
                df[f'{target_col}_rolling_std_{window}'] = df.groupby(group_col)[target_col].transform(
                    lambda x: x.rolling(window=window, min_periods=1).std()
                )
                
                # Rolling min
                df[f'{target_col}_rolling_min_{window}'] = df.groupby(group_col)[target_col].transform(
                    lambda x: x.rolling(window=window, min_periods=1).min()
                )
                
                # Rolling max
                df[f'{target_col}_rolling_max_{window}'] = df.groupby(group_col)[target_col].transform(
                    lambda x: x.rolling(window=window, min_periods=1).max()
                )
                
                # Rolling range
                df[f'{target_col}_rolling_range_{window}'] = (
                    df[f'{target_col}_rolling_max_{window}'] - 
                    df[f'{target_col}_rolling_min_{window}']
                )
        else:
            for window in windows:
                df[f'{target_col}_rolling_mean_{window}'] = df[target_col].rolling(window=window, min_periods=1).mean()
                df[f'{target_col}_rolling_std_{window}'] = df[target_col].rolling(window=window, min_periods=1).std()
        
        print(f"Created {len(windows) * 5} rolling features")
        
        return df
    
    def create_diff_features(self, df, target_col, periods=[1, 24, 168],
                           group_col='SITE_ID'):
        """Create differencing features"""
        
        if target_col not in df.columns:
            print(f"Target column {target_col} not found")
            return df
        
        print(f"\nCreating differencing features for {target_col}...")
        print(f"   Periods: {periods}")
        print("-" * 80)
        
        df = df.copy()
        
        if group_col in df.columns:
            for period in periods:
                df[f'{target_col}_diff_{period}'] = df.groupby(group_col)[target_col].diff(periods=period)
        else:
            for period in periods:
                df[f'{target_col}_diff_{period}'] = df[target_col].diff(periods=period)
        
        print(f"Created {len(periods)} differencing features")
        
        return df
    
    def create_rate_of_change(self, df, target_col, group_col='SITE_ID'):
        """Calculate rate of change (velocity and acceleration)"""
        
        if target_col not in df.columns:
            print(f"Target column {target_col} not found")
            return df
        
        print(f"\nCreating rate of change features for {target_col}...")
        print("-" * 80)
        
        df = df.copy()
        
        # First derivative (velocity)
        if group_col in df.columns:
            df[f'{target_col}_velocity'] = df.groupby(group_col)[target_col].diff(1)
            df[f'{target_col}_acceleration'] = df.groupby(group_col)[f'{target_col}_velocity'].diff(1)
        else:
            df[f'{target_col}_velocity'] = df[target_col].diff(1)
            df[f'{target_col}_acceleration'] = df[f'{target_col}_velocity'].diff(1)
        
        print(f"Created 2 rate of change features")
        
        return df


SECTION 5: STATISTICAL FEATURES

In [6]:
class StatisticalFeatureEngineer:
    """Create statistical features"""
    
    def __init__(self):
        pass
    
    def create_expanding_features(self, df, target_col, group_col='SITE_ID'):
        """Create expanding window features (cumulative statistics)"""
        
        if target_col not in df.columns:
            print(f"Target column {target_col} not found")
            return df
        
        print(f"\nCreating expanding window features for {target_col}...")
        print("-" * 80)
        
        df = df.copy()
        
        if group_col in df.columns:
            # Expanding mean
            df[f'{target_col}_expanding_mean'] = df.groupby(group_col)[target_col].transform(
                lambda x: x.expanding(min_periods=1).mean()
            )
            
            # Expanding std
            df[f'{target_col}_expanding_std'] = df.groupby(group_col)[target_col].transform(
                lambda x: x.expanding(min_periods=1).std()
            )
        else:
            df[f'{target_col}_expanding_mean'] = df[target_col].expanding(min_periods=1).mean()
            df[f'{target_col}_expanding_std'] = df[target_col].expanding(min_periods=1).std()
        
        print(f"Created 2 expanding window features")
        
        return df
    
    def create_percentile_features(self, df, target_col, group_col='SITE_ID'):
        """Create percentile-based features"""
        
        if target_col not in df.columns or group_col not in df.columns:
            print(f"Required columns not found")
            return df
        
        print(f"\nCreating percentile features for {target_col}...")
        print("-" * 80)
        
        df = df.copy()
        
        # Site-level percentiles
        site_stats = df.groupby(group_col)[target_col].agg([
            ('p25', lambda x: x.quantile(0.25)),
            ('p50', lambda x: x.quantile(0.50)),
            ('p75', lambda x: x.quantile(0.75)),
            ('p90', lambda x: x.quantile(0.90))
        ]).reset_index()
        
        df = df.merge(site_stats, on=group_col, how='left', suffixes=('', '_site'))
        
        # Relative position within site distribution
        if target_col in df.columns:
            df[f'{target_col}_above_site_median'] = (df[target_col] > df['p50']).astype(int)
            df[f'{target_col}_above_site_p75'] = (df[target_col] > df['p75']).astype(int)
        
        print(f"Created percentile features")
        
        return df


SECTION 6: MISSING DATA HANDLING

In [7]:
class MissingDataHandler:
    def analyze_missing_from_chunks(self, chunks):
        """
        Analyze missing data across many chunks without concatenating
        into a single giant DataFrame.
        """
        total_rows = 0
        missing_counts = None
        dtypes = None

        for i, df in enumerate(chunks, start=1):
            # Only look at numeric columns
            numeric_cols = df.select_dtypes(include=[np.number]).columns
            df_num = df[numeric_cols]

            if missing_counts is None:
                missing_counts = df_num.isna().sum()
                dtypes = df_num.dtypes
            else:
                missing_counts = missing_counts.add(df_num.isna().sum(), fill_value=0)

            total_rows += len(df_num)
            print(f"Processed chunk {i}, rows so far: {total_rows:,}")

        missing_pct = (missing_counts / total_rows) * 100

        missing_info = (
            pd.DataFrame({
                "missing_count": missing_counts,
                "missing_pct": missing_pct,
                "dtype": dtypes,
            })
            .sort_values("missing_pct", ascending=False)
        )

        print("=" * 80)
        print("Missing data summary (numeric columns):")
        print(missing_info.head(20))

        return missing_info

    
    def create_complete_time_index(self, df, site_col='SITE_ID', 
                                   time_col='DATE_TIME', freq='H'):
        """Create complete time index for each site"""
        
        if site_col not in df.columns or time_col not in df.columns:
            print(f"Required columns not found")
            return df
        
        print(f"\nCreating complete time index (freq={freq})...")
        print("-" * 80)
        
        result_dfs = []
        
        sites = df[site_col].unique()
        print(f"Processing {len(sites)} sites...")
        
        for i, site in enumerate(sites):
            if (i + 1) % 10 == 0 or i == 0:
                print(f"   Processing site {i+1}/{len(sites)}")
            
            site_data = df[df[site_col] == site].copy()
            
            # Create complete time range
            start = site_data[time_col].min()
            end = site_data[time_col].max()
            complete_index = pd.date_range(start=start, end=end, freq=freq)
            
            # Reindex
            site_data = site_data.set_index(time_col)
            site_data = site_data.reindex(complete_index)
            site_data[site_col] = site
            site_data.index.name = time_col
            site_data = site_data.reset_index()
            
            result_dfs.append(site_data)
        
        df_complete = pd.concat(result_dfs, ignore_index=True)
        
        print(f"Complete index created: {len(df_complete):,} records")
        
        return df_complete
    
    def interpolate_missing(self, df, method='time', limit=3):
        """Interpolate missing values"""
        
        print(f"\nInterpolating missing values (method={method}, limit={limit})...")
        print("-" * 80)
        
        df_filled = df.copy()
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        
        before_missing = df[numeric_cols].isna().sum().sum()
        
        for col in numeric_cols:
            if df[col].isna().any():
                df_filled[col] = df[col].interpolate(
                    method=method if method != 'time' else 'linear',
                    limit=limit,
                    limit_direction='both'
                )
        
        after_missing = df_filled[numeric_cols].isna().sum().sum()
        
        print(f"Before: {before_missing:,} missing values")
        print(f"After:  {after_missing:,} missing values")
        print(f"Filled: {before_missing - after_missing:,} values")
        
        return df_filled


SECTION 7: OUTLIER DETECTION AND HANDLING

In [8]:
class OutlierHandler:
    """Detect and handle outliers"""
    
    def __init__(self):
        pass
    
    def detect_outliers_iqr(self, df, columns, multiplier=3.0):
        """Detect outliers using IQR method"""
        
        print(f"\nDetecting outliers (IQR method, multiplier={multiplier})...")
        print("-" * 80)
        
        outlier_mask = pd.Series(False, index=df.index)
        outlier_info = {}
        
        for col in columns:
            if col not in df.columns:
                continue
            
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            
            lower_bound = Q1 - multiplier * IQR
            upper_bound = Q3 + multiplier * IQR
            
            col_outliers = (df[col] < lower_bound) | (df[col] > upper_bound)
            outlier_mask = outlier_mask | col_outliers
            
            outlier_count = col_outliers.sum()
            outlier_pct = (outlier_count / len(df)) * 100
            
            outlier_info[col] = {
                'count': outlier_count,
                'percentage': outlier_pct,
                'lower_bound': lower_bound,
                'upper_bound': upper_bound
            }
            
            print(f"{col:30s}: {outlier_count:6d} outliers ({outlier_pct:5.2f}%)")
        
        total_outliers = outlier_mask.sum()
        print(f"\n   Total rows with outliers: {total_outliers:,} ({total_outliers/len(df)*100:.2f}%)")
        
        return outlier_mask, outlier_info
    
    def handle_outliers(self, df, outlier_mask, method='clip'):
        """Handle outliers"""
        
        print(f"\nHandling outliers (method={method})...")
        print("-" * 80)
        
        df_clean = df.copy()
        
        if method == 'remove':
            df_clean = df_clean[~outlier_mask]
            print(f"   Removed {outlier_mask.sum():,} rows")
        
        elif method == 'nan':
            numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
            df_clean.loc[outlier_mask, numeric_cols] = np.nan
            print(f"   Set {outlier_mask.sum():,} rows to NaN")
        
        elif method == 'clip':
            numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
            for col in numeric_cols:
                if df_clean[col].notna().any():
                    p01 = df_clean[col].quantile(0.01)
                    p99 = df_clean[col].quantile(0.99)
                    df_clean[col] = df_clean[col].clip(lower=p01, upper=p99)
            print(f"Clipped values to 1st-99th percentile range")
        
        return df_clean


SECTION 8: EXECUTE FEATURE ENGINEERING PIPELINE

In [9]:
# Initialize feature engineers
temporal_fe = TemporalFeatureEngineer()
lag_rolling_fe = LagRollingFeatureEngineer()
statistical_fe = StatisticalFeatureEngineer()
missing_handler = MissingDataHandler()
outlier_handler = OutlierHandler()

# Start with ozone data
df_features = df_ozone.copy()

print(f"\nStarting shape: {df_features.shape}")

# Downcast floats
float_cols = df_features.select_dtypes(include=['float64']).columns
df_features[float_cols] = df_features[float_cols].astype('float32')

# Downcast integers (if any)
int_cols = df_features.select_dtypes(include=['int64']).columns
df_features[int_cols] = df_features[int_cols].astype('int32')

# Verify memory usage
print(df_features.info(memory_usage='deep'))


# Step 1: Add temporal features
df_features = temporal_fe.add_temporal_features(df_features)

# Step 2: Create complete time index
if 'SITE_ID' in df_features.columns and 'DATE_TIME' in df_features.columns:
    df_features = missing_handler.create_complete_time_index(df_features)

# Step 3: Create lag features
if 'OZONE' in df_features.columns:
    df_features = lag_rolling_fe.create_lag_features(
        df_features, 
        'OZONE', 
        lags=[1, 2, 3, 6, 12, 24, 48, 168]  # Up to 1 week
    )

# Step 4: Create rolling features
if 'OZONE' in df_features.columns:
    df_features = lag_rolling_fe.create_rolling_features(
        df_features,
        'OZONE',
        windows=[3, 6, 12, 24, 168]  # Up to 1 week
    )

# Step 5: Create differencing features
if 'OZONE' in df_features.columns:
    df_features = lag_rolling_fe.create_diff_features(
        df_features,
        'OZONE',
        periods=[1, 24, 168]
    )

# Step 6: Create rate of change features
if 'OZONE' in df_features.columns:
    df_features = lag_rolling_fe.create_rate_of_change(df_features, 'OZONE')

# Step 7: Create statistical features
if 'OZONE' in df_features.columns and 'SITE_ID' in df_features.columns:
    df_features = statistical_fe.create_percentile_features(df_features, 'OZONE')

# Step 8: Merge with site metadata
if not df_site.empty and 'SITE_ID' in df_features.columns:
    print("\n Merging with site metadata...")
    print("-" * 80)
    
    # Select relevant site features
    site_features = ['SITE_ID', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 
                    'LAND_USE', 'TERRAIN', 'STATE']
    available_site_features = [f for f in site_features if f in df_site.columns]
    
    if available_site_features:
        df_features = df_features.merge(
            df_site[available_site_features],
            on='SITE_ID',
            how='left'
        )
        print(f"Merged {len(available_site_features)} site features")

processed_chunks = []

# Process each site individually
for site_id, group_df in df_features.groupby('SITE_ID'):
    group_df = lag_rolling_fe.create_diff_features(group_df, 'OZONE', periods=[1, 24, 168])
    processed_chunks.append(group_df)

# Combine back together (this might still be heavy, but usually works better)
df_features = pd.concat(processed_chunks)

# Step 9: Analyze missing data
missing_info = missing_handler.analyze_missing_from_chunks(processed_chunks)

# Step 10: Interpolate missing values
df_features = missing_handler.interpolate_missing(df_features, method='linear', limit=3)

# Step 11: Detect outliers (only for OZONE)
if 'OZONE' in df_features.columns:
    outlier_mask, outlier_info = outlier_handler.detect_outliers_iqr(
        df_features,
        ['OZONE'],
        multiplier=3.0
    )
    
    # Handle outliers
    df_features = outlier_handler.handle_outliers(
        df_features,
        outlier_mask,
        method='clip'
    )

print(f"\nFeature engineering complete!")
print(f"   Final shape: {df_features.shape}")
print(f"   Total features: {len(df_features.columns)}")



Starting shape: (21667030, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21667030 entries, 0 to 21667029
Data columns (total 10 columns):
 #   Column             Dtype         
---  ------             -----         
 0   SITE_ID            object        
 1   DATE_TIME          datetime64[ns]
 2   OZONE              float32       
 3   OZONE_F            float32       
 4   QA_CODE            int32         
 5   UPDATE_DATE        object        
 6   UNITS              float32       
 7   EXPECTED_VALUE     float32       
 8   CALIBRATION_VALUE  float32       
 9   CALIBRATION_TYPE   float32       
dtypes: datetime64[ns](1), float32(6), int32(1), object(2)
memory usage: 3.2 GB
None

Adding temporal features...
--------------------------------------------------------------------------------
Added 27 temporal features

Creating complete time index (freq=H)...
--------------------------------------------------------------------------------
Processing 127 sites...
   Processing si

MemoryError: Unable to allocate 8.56 GiB for an array with shape (47, 24443603) and data type float64

SECTION 9: SAVE ENGINEERED FEATURES

In [ ]:
print("\n" + "="*80)
print("SAVING ENGINEERED FEATURES")
print("="*80)

# Save full feature set
output_file = os.path.join(PROCESSED_DIR, 'features_engineered.csv')
df_features.to_csv(output_file, index=False)
print(f"Saved: {output_file}")

# Save feature list
feature_list = df_features.columns.tolist()
feature_list_file = os.path.join(PROCESSED_DIR, 'feature_list.txt')
with open(feature_list_file, 'w') as f:
    f.write("FEATURE LIST\n")
    f.write("="*80 + "\n\n")
    f.write(f"Total Features: {len(feature_list)}\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    
    # Categorize features
    temporal_features = [f for f in feature_list if any(x in f.lower() for x in ['year', 'month', 'day', 'hour', 'week', 'season', 'sin', 'cos', 'weekend', 'rush'])]
    lag_features = [f for f in feature_list if 'lag' in f.lower()]
    rolling_features = [f for f in feature_list if 'rolling' in f.lower()]
    diff_features = [f for f in feature_list if 'diff' in f.lower() or 'velocity' in f.lower() or 'acceleration' in f.lower()]
    stat_features = [f for f in feature_list if any(x in f.lower() for x in ['expanding', 'percentile', 'p25', 'p50', 'p75', 'p90', 'above'])]
    site_features = [f for f in feature_list if any(x in f.lower() for x in ['latitude', 'longitude', 'elevation', 'land', 'terrain', 'state'])]
    
    f.write(f"\nTEMPORAL FEATURES ({len(temporal_features)}):\n")
    for feat in temporal_features:
        f.write(f"  • {feat}\n")
    
    f.write(f"\nLAG FEATURES ({len(lag_features)}):\n")
    for feat in lag_features:
        f.write(f"  • {feat}\n")
    
    f.write(f"\nROLLING WINDOW FEATURES ({len(rolling_features)}):\n")
    for feat in rolling_features:
        f.write(f"  • {feat}\n")
    
    f.write(f"\nDIFFERENCING FEATURES ({len(diff_features)}):\n")
    for feat in diff_features:
        f.write(f"  • {feat}\n")
    
    f.write(f"\nSTATISTICAL FEATURES ({len(stat_features)}):\n")
    for feat in stat_features:
        f.write(f"  • {feat}\n")
    
    f.write(f"\nSITE FEATURES ({len(site_features)}):\n")
    for feat in site_features:
        f.write(f"  • {feat}\n")
    
    f.write(f"\nOTHER FEATURES:\n")
    other_features = [f for f in feature_list if f not in temporal_features + lag_features + rolling_features + diff_features + stat_features + site_features]
    for feat in other_features:
        f.write(f"  • {feat}\n")

print(f"Saved feature list: {feature_list_file}")

print(f"\nAll files saved to: {PROCESSED_DIR}")
